In [2]:
# ============================================================
# SALES FORECASTING USING LSTM (DEEP LEARNING)
# ============================================================

# ======================
# INSTALL REQUIRED LIBRARIES
# ======================

# pip install tensorflow
# pip install scikit-learn

# ======================
# IMPORT LIBRARIES
# ======================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

# ======================
# LOAD DATASET
# ======================

df = pd.read_csv("eda_final_superstore.csv")

# ======================
# DATE PROCESSING
# ======================

df['Order Date'] = pd.to_datetime(df['Order Date'])

# ======================
# MONTHLY SALES
# ======================

monthly_sales = df.groupby(
    pd.Grouper(key='Order Date', freq='M')
)['Sales'].sum().reset_index()

# ======================
# PREPARE DATA
# ======================

sales_data = monthly_sales['Sales'].values.reshape(-1,1)

# ======================
# NORMALIZATION
# ======================

scaler = MinMaxScaler(feature_range=(0,1))

scaled_data = scaler.fit_transform(sales_data)

# ======================
# CREATE SEQUENCES
# ======================

X = []
y = []

sequence_length = 3

for i in range(sequence_length, len(scaled_data)):
    
    X.append(scaled_data[i-sequence_length:i, 0])
    
    y.append(scaled_data[i, 0])

X = np.array(X)
y = np.array(y)

# ======================
# RESHAPE FOR LSTM
# ======================

X = np.reshape(
    X,
    (X.shape[0], X.shape[1], 1)
)

# ======================
# TRAIN TEST SPLIT
# ======================

split = int(len(X) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

# ======================
# BUILD LSTM MODEL
# ======================

model = Sequential()

model.add(
    LSTM(
        units=64,
        return_sequences=True,
        input_shape=(X_train.shape[1],1)
    )
)

model.add(Dropout(0.2))

model.add(
    LSTM(
        units=64
    )
)

model.add(Dropout(0.2))

model.add(Dense(1))

# ======================
# COMPILE MODEL
# ======================

model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

# ======================
# TRAIN MODEL
# ======================

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=4,
    validation_data=(X_test, y_test),
    verbose=1
)

# ======================
# PREDICTIONS
# ======================

predictions = model.predict(X_test)

# ======================
# INVERSE TRANSFORM
# ======================

predictions = scaler.inverse_transform(predictions.reshape(-1,1))

y_test_actual = scaler.inverse_transform(
    y_test.reshape(-1,1)
)

# ======================
# VISUALIZATION
# ======================

plt.figure(figsize=(12,6))

plt.plot(
    y_test_actual,
    label='Actual Sales'
)

plt.plot(
    predictions,
    label='Predicted Sales'
)

plt.title("LSTM Sales Forecasting")
plt.xlabel("Time")
plt.ylabel("Sales")

plt.legend()

plt.show()

# ======================
# FUTURE FORECAST
# ======================

last_sequence = scaled_data[-sequence_length:]

future_forecast = []

current_sequence = last_sequence

for _ in range(6):

    current_sequence_reshaped = np.reshape(
        current_sequence,
        (1, sequence_length, 1)
    )

    next_prediction = model.predict(
        current_sequence_reshaped,
        verbose=0
    )

    future_forecast.append(next_prediction[0,0])

    current_sequence = np.append(
        current_sequence[1:],
        next_prediction
    )

future_forecast = np.array(future_forecast).reshape(-1,1)

future_forecast = scaler.inverse_transform(
    future_forecast
)

# ======================
# DISPLAY FORECAST
# ======================

print("\nFuture Sales Forecast:\n")

for i, value in enumerate(future_forecast, start=1):
    
    print(f"Month {i}: {value[0]:.2f}")

# ======================
# SAVE FORECAST
# ======================

forecast_df = pd.DataFrame({
    'Forecast_Month': range(1,7),
    'Predicted_Sales': future_forecast.flatten()
})

forecast_df.to_csv(
    "sales_forecast.csv",
    index=False
)

print("\nLSTM Sales Forecasting Completed Successfully.")

ModuleNotFoundError: No module named 'tensorflow'